# Pydantic is the new Cypher

This notebook will illustrate the challenge of text-to-Cypher approaches with LLMs,

> and why we believe **structured outputs with Pydantic** is the best way to query databases with LLMs.

# Memgraph Setup

In [4]:
from neo4j import GraphDatabase
 
# Define correct URI and AUTH arguments (no AUTH by default)
URI = "bolt://localhost:7687"
AUTH = ("", "")
 
with GraphDatabase.driver(URI, auth=AUTH) as client:
    # Check the connection
    client.verify_connectivity()
 
    # Create a user in the database
    records, summary, keys = client.execute_query(
        "CREATE (u:User {name: $name, password: $password}) RETURN u.name AS name;",
        name="John",
        password="pass",
        database_="memgraph",
    )
 
    # Get the result
    for record in records:
        print(record["name"])
 
    # Print the query counters
    print(summary.counters)
 
    # Find a user John in the database
    records, summary, keys = client.execute_query(
        "MATCH (u:User {name: $name}) RETURN u.name AS name",
        name="John",
        database_="memgraph",
    )
 
    # Get the result
    for record in records:
        print(record["name"])
 
    # Print the query
    print(summary.query)

John
{'labels_added': 1, 'labels_removed': 0, 'nodes_created': 1, 'nodes_deleted': 0, 'properties_set': 0, 'relationships_created': 0, 'relationships_deleted': 0}
John
MATCH (u:User {name: $name}) RETURN u.name AS name


In [5]:
# Import and setup

# Create Companies
for company_data in [
    ("Memgraph", 2016),
    ("Neo4j", 2007),
    ("AgensGraph", 2016),
    ("TigerGraph", 2012)
]:
    records, summary, keys = client.execute_query(
        "CREATE (c:Company {name: $name, founded: $founded})",
        name=company_data[0],
        founded=company_data[1],
        database_="memgraph"
    )

# Create Topics
for topic_data in [
    ("Transactions", "Database Core"),
    ("Backup Systems", "Operations"),
    ("Query Languages", "User Interface"),
    ("Graph Algorithms", "Analytics"),
    ("Storage Engines", "Infrastructure"),
    ("Distributed Systems", "Infrastructure")
]:
    records, summary, keys = client.execute_query(
        "CREATE (t:Topic {name: $name, field: $field})",
        name=topic_data[0],
        field=topic_data[1],
        database_="memgraph"
    )

# Create People
for person_data in [
    ("Alice Chen", "Database Engineer", 8),
    ("Bob Smith", "Systems Architect", 12),
    ("Carol Kumar", "Research Engineer", 5),
    ("David Garcia", "Technical Writer", 6),
    ("Elena Wilson", "Performance Engineer", 9)
]:
    records, summary, keys = client.execute_query(
        "CREATE (p:Person {name: $name, role: $role, yearsExp: $exp})",
        name=person_data[0],
        role=person_data[1],
        exp=person_data[2],
        database_="memgraph"
    )

# Create Employment Relationships
for employment_data in [
    ("Alice Chen", "Memgraph", 2019),
    ("Bob Smith", "Neo4j", 2015),
    ("Carol Kumar", "TigerGraph", 2020),
    ("David Garcia", "AgensGraph", 2018),
    ("Elena Wilson", "Neo4j", 2017)
]:
    records, summary, keys = client.execute_query(
        """
        MATCH (p:Person), (c:Company)
        WHERE p.name = $person_name AND c.name = $company_name
        CREATE (p)-[:WORKS_AT {since: $since}]->(c)
        """,
        person_name=employment_data[0],
        company_name=employment_data[1],
        since=employment_data[2],
        database_="memgraph"
    )

# Create Expertise Relationships
for expertise_data in [
    ("Alice Chen", "Transactions", "Advanced", 5),
    ("Alice Chen", "Storage Engines", "Intermediate", 3),
    ("Bob Smith", "Distributed Systems", "Advanced", 8)
]:
    records, summary, keys = client.execute_query(
        """
        MATCH (p:Person), (t:Topic)
        WHERE p.name = $person_name AND t.name = $topic_name
        CREATE (p)-[:EXPERT_IN {level: $level, years: $years}]->(t)
        """,
        person_name=expertise_data[0],
        topic_name=expertise_data[1],
        level=expertise_data[2],
        years=expertise_data[3],
        database_="memgraph"
    )

# Create Company Focus Areas
for focus_data in [
    ("Memgraph", "Graph Algorithms"),
    ("Neo4j", "Transactions"),
    ("TigerGraph", "Graph Algorithms")
]:
    records, summary, keys = client.execute_query(
        """
        MATCH (c:Company), (t:Topic)
        WHERE c.name = $company_name AND t.name = $topic_name
        CREATE (c)-[:FOCUSES_ON {priority: 'High'}]->(t)
        """,
        company_name=focus_data[0],
        topic_name=focus_data[1],
        database_="memgraph"
    )

# Verify the data
records, summary, keys = client.execute_query(
    """
    RETURN p.name as person, type(r) as relationship, n.name as target
    """,
    database_="memgraph"
)

print("\nVerifying relationships:")
for record in records:
    print(f"{record['person']} -{record['relationship']}-> {record['target']}")


Verifying relationships:
Alice Chen -WORKS_AT-> Memgraph
Alice Chen -EXPERT_IN-> Transactions
Alice Chen -EXPERT_IN-> Storage Engines
Bob Smith -WORKS_AT-> Neo4j
Bob Smith -EXPERT_IN-> Distributed Systems
Carol Kumar -WORKS_AT-> TigerGraph
David Garcia -WORKS_AT-> AgensGraph
Elena Wilson -WORKS_AT-> Neo4j


/var/folders/41/8dp_379x15d8zz4ppsjthdw40000gn/T/ipykernel_24994/2828091927.py:10: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = client.execute_query(
/var/folders/41/8dp_379x15d8zz4ppsjthdw40000gn/T/ipykernel_24994/2828091927.py:26: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = client.execute_query(
/var/folders/41/8dp_379x15d8zz4ppsjthdw40000gn/T/ipykernel_24994/2828091927.py:41: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = client.execute_query(
/var/folders/41/8dp_379x15d8zz4ppsjthdw40000gn/T/ipykernel_24994/2828091927.py:57: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summar

In [6]:
records, summary, keys = client.execute_query(
    """
    MATCH (p:Person)-[:WORKS_AT]->(c:Company {name: 'Neo4j'})
    OPTIONAL MATCH (p)-[e:EXPERT_IN]->(t:Topic)
    RETURN p.name as person, p.role as role, t.name as expertise, e.level as level
    """,
    database_="memgraph"
)

print("Neo4j Employees and their expertise:")
for record in records:
    expertise = f"{record['expertise']} ({record['level']})" if record['expertise'] else "No expertise listed"
    print(f"{record['person']} ({record['role']}) - {expertise}")

Neo4j Employees and their expertise:
Bob Smith (Systems Architect) - Distributed Systems (Advanced)
Elena Wilson (Performance Engineer) - No expertise listed


/var/folders/41/8dp_379x15d8zz4ppsjthdw40000gn/T/ipykernel_24994/4176022777.py:1: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = client.execute_query(


# DSPy setup

In [12]:
import dspy
import os
lm = dspy.LM('openai/gpt-4o', api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

In [13]:
lm("What is the future of database systems with AI? In 3 words.")

['Intelligent, automated, scalable.']

# Pydantic is the new Cypher

In [15]:
from pydantic import BaseModel
from typing import Optional, List, Tuple, Union

# The Cypher data model you provided earlier:
class CypherFilter(BaseModel):
    """
    Represents a single condition in the WHERE clause.
    """
    field: str
    operator: str
    value: Union[str, int, float]

class CypherQuery(BaseModel):
    """
    A basic model for generating Cypher (Memgraph) queries, focusing on:
    - MATCH (n:Label)
    - Optional WHERE
    - RETURN
    - Optional ORDER BY
    - Optional LIMIT
    """
    label: str
    fields: List[str]
    filters: Optional[List[CypherFilter]] = None
    limit: Optional[int] = None
    sort_by: Optional[List[Tuple[str, str]]] = None  # e.g. [("age", "DESC")]

    def to_cypher(self) -> str:
        query = f"MATCH (n:{self.label})"

        if self.filters:
            conditions = []
            for f in self.filters:
                val = f"\"{f.value}\"" if isinstance(f.value, str) else str(f.value)
                conditions.append(f"n.{f.field} {f.operator} {val}")
            query += " WHERE " + " AND ".join(conditions)

        if not self.fields:
            query += " RETURN n"
        else:
            fields_str = ", ".join([f"n.{field}" for field in self.fields])
            query += f" RETURN {fields_str}"

        if self.sort_by:
            sort_clauses = [f"n.{field} {direction}" for field, direction in self.sort_by]
            query += " ORDER BY " + ", ".join(sort_clauses)

        if self.limit is not None:
            query += f" LIMIT {self.limit}"

        return query

# DSPy `MemgraphQueryWriter`

In [16]:
class MemgraphQueryWriter(dspy.Signature):
    """
    Translate a natural language information need into a Memgraph (Cypher) query.
    
    In your rationale for generating the final query string, you should be 
    very clear about why you chose specific query operators (e.g., WHERE 
    clauses, sort directions, etc.) and why you do not need the operators 
    that you chose not to include.
    """
    
    # The natural language command from the user
    nl_command: str = dspy.InputField(
        desc="A natural language command with an underlying information need your db_query should answer."
    )
    
    # The database schema or relevant metadata
    db_schema: str = dspy.InputField(
        desc="The database schema (Memgraph schema) you can query."
    )

    # The resulting Cypher query object (built to meet the info need)
    db_query: CypherQuery = dspy.OutputField(
        desc="A CypherQuery model that captures how to query Memgraph."
    )

In [22]:
records, summary, keys = client.execute_query(
    "MATCH (n) RETURN DISTINCT labels(n) AS labels",
    database_="memgraph"
)
node_labels = []
for record in records:
    node_labels.extend(record["labels"])
node_labels_str = ", ".join(node_labels)
print(f"Node labels: {node_labels_str}")


Node labels: User, Company, Topic, Person


/var/folders/41/8dp_379x15d8zz4ppsjthdw40000gn/T/ipykernel_24994/2423293021.py:1: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  records, summary, keys = client.execute_query(


In [24]:
memgraph_writer = dspy.ChainOfThought(MemgraphQueryWriter)

generated_query = memgraph_writer(
    nl_command = "How many people work at Memgraph?",
    db_schema = node_labels_str
)

print(generated_query)
db_query = generated_query.db_query

Prediction(
    reasoning='The natural language command asks for the number of people working at Memgraph. In the given database schema, we have entities like User, Company, Topic, and Person. To find out how many people work at Memgraph, we need to focus on the `Person` label, as it is the most likely to represent individuals. We assume there is a relationship or property that connects `Person` to `Company`, specifically to Memgraph. However, since the schema does not provide explicit relationships or properties, we will assume that there is a property or relationship that can be used to filter `Person` nodes associated with Memgraph. The query will count the number of `Person` nodes that are associated with Memgraph.',
    db_query=CypherQuery(label='Person', fields=['count(*)'], filters=[CypherFilter(field='company', operator='=', value='Memgraph')], limit=None, sort_by=None)
)
